In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import glob
import os
import random
import csv
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd

# ==========================================
# ⚙️ 1. CONFIGURATION & REPRODUCIBILITY
# ==========================================
DATA_DIR = "3d_unet_grid_patches" 
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_EPOCHS = 30 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=42):
    """Locks all random seeds for full reproducibility."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

def save_final_plot(csv_path):
    df = pd.read_csv(csv_path)
    plt.figure(figsize=(10, 5))
    plt.plot(df['Epoch'], df['Train_Z_Loss'], label='Train Loss (Z-Score)')
    plt.plot(df['Epoch'], df['Val_Real_MAE'], label='Val MAE (Physical)')
    plt.xlabel('Epoch')
    plt.ylabel('Error')
    plt.legend()
    plt.title('Final Training Evidence')
    plt.savefig('training_evidence_plot.png')
    print("📈 Evidence plot saved to training_evidence_plot.png")
    plt.close()


# ==========================================
# 📦 2. DATASET
# ==========================================
class Satellite3DDataset(Dataset):
    def __init__(self, data_dir, seq_len=16, mode='train', 
                 sat_channels=[0, 1], epoch_len_multiplier=1, stats=None):
        
        all_files = sorted(glob.glob(os.path.join(data_dir, "*.npz")))
        if not all_files:
            abs_dir = os.path.abspath(data_dir)
            raise FileNotFoundError(
                f"No .npz files found in '{data_dir}' (resolved: {abs_dir}). "
                "Check DATA_DIR casing/path and your working directory."
            )
        self.files = []
        
        train_grids = {1, 2, 9, 10, 11, 12}
        val_grids   = {4, 8}
        test_grids  = {3, 13}
        target_grids = train_grids if mode=='train' else val_grids if mode=='val' else test_grids
        
        print(f"[{mode.upper()}] Scanning files...")
        for f in all_files:
            try:
                grid_id = int(os.path.basename(f).split('_')[1])
                if grid_id in target_grids: self.files.append(f)
            except: pass
            
        if len(self.files) == 0:
            raise ValueError(
                f"No files matched split '{mode}'. "
                "Check filename grid-id parsing and split definitions."
            )
        self.cache = []
        if len(self.files) > 0:
            print(f"[{mode.upper()}] Caching {len(self.files)} files...")
            for f in tqdm(self.files):
                with np.load(f) as data:
                    self.cache.append(data['data'])
                    
        self.mode = mode
        self.seq_len = seq_len
        self.sat_channels = sat_channels
        self.epoch_len_multiplier = epoch_len_multiplier
        self.MIN_LAT = 49.0
        self.MAX_LAT = 61.0

        if stats is None:
            self.b4_m, self.b4_s, self.b8_m, self.b8_s = self._compute_stats()
        else:
            self.b4_m, self.b4_s, self.b8_m, self.b8_s = stats

    def _compute_stats(self):
        print(f"📊 Computing Z-Score Statistics deterministically...")
        b4_vals, b8_vals = [], []
        for chip in self.cache:
            b4 = chip[:, :, :, 0]
            b8 = chip[:, :, :, 1]
            valid_b4 = b4[~np.isnan(b4)]
            valid_b8 = b8[~np.isnan(b8)]
            
            if len(valid_b4) > 5000:
                step_b4 = len(valid_b4) // 5000
                valid_b4 = valid_b4[::step_b4][:5000]
            if len(valid_b8) > 5000:
                step_b8 = len(valid_b8) // 5000
                valid_b8 = valid_b8[::step_b8][:5000]
                
            if valid_b4.size > 0:
                b4_vals.append(valid_b4)
            if valid_b8.size > 0:
                b8_vals.append(valid_b8)
            
        if not b4_vals or not b8_vals:
            raise ValueError(
                "No valid B4/B8 pixels found while computing statistics. "
                "Dataset may be empty or contain only NaNs in channels 0/1."
            )
        b4_all = np.concatenate(b4_vals)
        b8_all = np.concatenate(b8_vals)
        
        b4_m, b4_s = float(b4_all.mean()), float(b4_all.std())
        b8_m, b8_s = float(b8_all.mean()), float(b8_all.std())
        
        print(f"   ➤ Red (B4) Mean: {b4_m:.4f} | Std: {b4_s:.4f} (Pixels: {len(b4_all)})")
        print(f"   ➤ NIR (B8) Mean: {b8_m:.4f} | Std: {b8_s:.4f} (Pixels: {len(b8_all)})")
        return b4_m, b4_s, b8_m, b8_s

    def get_stats(self):
        return (self.b4_m, self.b4_s, self.b8_m, self.b8_s)

    def forward_fill_numpy(self, arr):
        mask = np.isnan(arr)
        out = arr.copy()
        for t in range(1, out.shape[0]):
            missing = mask[t]
            out[t][missing] = out[t-1][missing]
        mask_rev = np.isnan(out)
        if np.any(mask_rev):
            for t in range(out.shape[0]-2, -1, -1):
                missing = mask_rev[t]
                out[t][missing] = out[t+1][missing]
        return np.nan_to_num(out, nan=0.0)

    def __len__(self):
        return len(self.files) * self.epoch_len_multiplier

    def __getitem__(self, idx):
            # 1. Standardize Index and Cache Access
            actual_idx = idx % len(self.files)
            full_seq = self.cache[actual_idx]
        
            # 2. Temporal Window Selection
            total_time = full_seq.shape[0]
            if total_time < self.seq_len:
                reps = int(np.ceil(self.seq_len / total_time))
                full_seq = np.tile(full_seq, (reps, 1, 1, 1))
                total_time = full_seq.shape[0]
        
            start_t = np.random.randint(0, total_time - self.seq_len + 1)
            seq_raw = full_seq[start_t : start_t + self.seq_len].copy()
        
            # 3. CAPTURE PHYSICAL TRUTH (Raw Validity)
            # Identify exactly where real photons were captured by the sensor
            real_target_mask_np = ~np.isnan(seq_raw[:, :, :, 0:2])
            real_target_mask = torch.from_numpy(real_target_mask_np.copy()).permute(3, 0, 1, 2).float()

            # 4. Z-SCORE NORMALIZATION
            seq_raw[:, :, :, 0] = (seq_raw[:, :, :, 0] - self.b4_m) / (self.b4_s + 1e-8)
            seq_raw[:, :, :, 1] = (seq_raw[:, :, :, 1] - self.b8_m) / (self.b8_s + 1e-8)
        
            # Normalize Latitude (Channel 18)
            lat_channel = seq_raw[:, :, :, 18]
            seq_raw[:, :, :, 18] = (lat_channel - self.MIN_LAT) / (self.MAX_LAT - self.MIN_LAT)
        
            # 5. PHYSICS-AWARE AUGMENTATIONS (Train Mode Only)
            if self.mode == 'train':
                k = np.random.randint(0, 4)
                do_flip = np.random.rand() > 0.5
            
                if k > 0 or do_flip:
                    seq_raw = np.rot90(seq_raw, k=k, axes=(1, 2))
                    real_target_mask_np = np.rot90(real_target_mask_np, k=k, axes=(1, 2))
                    if do_flip:
                        seq_raw = np.flip(seq_raw, axis=2)
                        real_target_mask_np = np.flip(real_target_mask_np, axis=2)
                
                    # Trigonometric corrections for Aspect (Indices 10 & 11)
                    a_sin, a_cos = seq_raw[:, :, :, 10].copy(), seq_raw[:, :, :, 11].copy()
                    if k == 1: # 90 CCW
                        seq_raw[:, :, :, 10], seq_raw[:, :, :, 11] = -a_cos, a_sin
                    elif k == 2: # 180
                        seq_raw[:, :, :, 10], seq_raw[:, :, :, 11] = -a_sin, -a_cos
                    elif k == 3: # 270 CCW
                        seq_raw[:, :, :, 10], seq_raw[:, :, :, 11] = a_cos, -a_sin
                    if do_flip: 
                        seq_raw[:, :, :, 10] = -seq_raw[:, :, :, 10]

                # Re-convert real_target_mask after augmentations
                real_target_mask = torch.from_numpy(real_target_mask_np.copy()).permute(3, 0, 1, 2).float()

            # 6. INPUT PREPARATION (Imputed for Model Input Only)
            seq_filled = self.forward_fill_numpy(seq_raw)
            clean_input = torch.from_numpy(seq_filled).permute(3, 0, 1, 2).float()

            C, T, H, W = clean_input.shape
            cloud_mask = torch.ones((1, T, H, W))
        
            # 7. DECOUPLED MASKING POLICIES
            if self.mode == 'train':
                # Randomized holes for training generalizability
                is_tube_mode = np.random.rand() < 0.8
                if is_tube_mode:
                    # Gap distribution aligned to Transformer V5
                    # Computed from V5's independent hide_before/hide_after sampling:
                    #   hide ~ Categorical([0.25, 0.35, 0.25, 0.15])
                    #   gap = hide_before + 1 + hide_after
                    _GAP_PROBS = [0.0625, 0.175, 0.2475, 0.25, 0.1675, 0.075, 0.0225]
                    gap = np.random.choice(np.arange(1, 8), p=_GAP_PROBS)
                    st = np.random.randint(0, T - gap + 1)
                    sz = np.random.randint(24, 48)
                    tp, lf = np.random.randint(0, H - sz), np.random.randint(0, W - sz)
                    cloud_mask[:, st:st+gap, tp:tp+sz, lf:lf+sz] = 0.0
                else:
                    for t in range(T):
                        if np.random.rand() < 0.5:
                            sz = np.random.randint(16, 32)
                            tp, lf = np.random.randint(0, H - sz), np.random.randint(0, W - sz)
                            cloud_mask[:, t, tp:tp+sz, lf:lf+sz] = 0.0
            
                # Loss Mask: All synthetic hole pixels with real sensor data
                target_hole_mask = (cloud_mask[0:2] == 0.0).float() * real_target_mask
            
            elif self.mode == 'val' or self.mode == 'test':
                # 📏 VAL/TEST: FIXED Centered Benchmark for epoch-to-epoch comparison
                center_t = T // 2
                # Fixed 3-frame temporal blackout
                h_st, h_en = max(0, center_t - 1), min(T, center_t + 2)
            
                # Fixed 32x32 spatial blackout in the dead center
                sz = 32
                tp, lf = (H - sz) // 2, (W - sz) // 2
                
                cloud_mask[:, h_st:h_en, tp:tp+sz, lf:lf+sz] = 0.0
                
                # Val Hole Mask: ONLY evaluate the center 'Y' frame for the metric
                eval_mask = torch.zeros((2, T, H, W))
                eval_mask[:, center_t, tp:tp+sz, lf:lf+sz] = 1.0
                target_hole_mask = eval_mask * real_target_mask

            # 8. TRUE OBSERVATION MASK (Channel 20)
            # Intersect natural sensor validity with synthetic drop mask
            natural_valid = (real_target_mask[0] * real_target_mask[1]).unsqueeze(0)
            input_mask_channel = natural_valid.clone()
            input_mask_channel[cloud_mask[0:1] == 0.0] = 0.0
        
            # 9. FINAL INPUT & TARGET ASSEMBLY
            model_input = clean_input.clone()
            # Ensure model cannot see the reflectance values inside the synthetic holes
            model_input[0:2] = model_input[0:2] * cloud_mask[0:1]
        
            final_input = torch.cat([model_input, input_mask_channel], dim=0)
        
            # TARGET IS RAW (Z-scored but NOT filled)
            # Fill NaNs with 0.0 only for tensor compatibility; masked loss ignores them
            target_np = np.ascontiguousarray(seq_raw[:, :, :, 0:2])
            target_tensor = torch.from_numpy(target_np).permute(3, 0, 1, 2).float()
            target_tensor = torch.nan_to_num(target_tensor, nan=0.0)
        
            # Visible Mask for identity supervision
            target_visible_mask = (1.0 - (cloud_mask[0:2] == 0.0).float()) * real_target_mask

            return final_input, target_tensor, target_hole_mask, target_visible_mask
# ==========================================
# 🧠 3. MODEL ARCHITECTURE
# ==========================================
class HybridUNet3D(nn.Module):
    def __init__(self, in_channels=20): 
        super().__init__()
        self.input_norm = nn.BatchNorm3d(in_channels)
        self.enc1 = self.conv_block(in_channels, 32)
        self.pool1 = nn.MaxPool3d(2)
        self.enc2 = self.conv_block(32, 64)
        self.pool2 = nn.MaxPool3d(2)
        self.bottleneck = self.conv_block(64, 128)
        self.up2 = nn.ConvTranspose3d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(64 + 64, 64)
        self.up1 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(32 + 32, 32)
        self.head_b4 = nn.Conv3d(32, 1, kernel_size=1)
        self.head_b8 = nn.Conv3d(32, 1, kernel_size=1)
        
    def conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, 1, 1), nn.BatchNorm3d(out_c), nn.ReLU(),
            nn.Conv3d(out_c, out_c, 3, 1, 1), nn.BatchNorm3d(out_c), nn.ReLU()
        )
        
    def forward(self, x):
        x = self.input_norm(x)
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        b = self.bottleneck(p2)
        u2 = self.up2(b)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        out_b4 = self.head_b4(d1)
        out_b8 = self.head_b8(d1)
        return torch.cat([out_b4, out_b8], dim=1)

# ==========================================
# 🚀 4. TRAINING LOOP
# ==========================================
def run_training():
    print(f"\n🚀 STARTING MASTER TRAINING PIPELINE")
    print(f"   Config: {NUM_EPOCHS} Epochs | Z-Scores | Physics-Aware Aug | Val: 45-day Benchmark")
    print("="*60)

    # 🚨 Initialize the Immutable Empirical Ledger
    history_file = open("training_history_master.csv", "w", newline="")
    history_writer = csv.writer(history_file)
    history_writer.writerow(["Epoch", "Train_Z_Loss", "Val_Real_MAE", "Best_Model"])

    train_ds = Satellite3DDataset(DATA_DIR, mode='train', epoch_len_multiplier=50)
    dataset_stats = train_ds.get_stats() 
    val_ds = Satellite3DDataset(DATA_DIR, mode='val', stats=dataset_stats)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = HybridUNet3D(in_channels=20).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    
    criterion = nn.L1Loss(reduction='none') 
    
    best_val_hole_loss = float('inf')
    b4_m, b4_s, b8_m, b8_s = dataset_stats
    
    try:
        for epoch in range(NUM_EPOCHS):
            model.train()
            train_loss = 0
            loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
            
            for inputs, targets, hole_mask, vis_mask in loop:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                hole_mask, vis_mask = hole_mask.to(DEVICE), vis_mask.to(DEVICE)
                
                outputs = model(inputs)
                raw_loss = criterion(outputs, targets[:, 0:2])
                
                l_hole = (raw_loss * hole_mask).sum() / (hole_mask.sum() + 1e-8)
                l_vis = (raw_loss * vis_mask).sum() / (vis_mask.sum() + 1e-8)
                
                loss = (l_hole * 10.0) + (l_vis * 1.0)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
                loop.set_postfix(loss=loss.item())
                
            model.eval()
            val_hole_loss_real = 0
            with torch.no_grad():
                for inputs, targets, hole_mask, vis_mask in val_loader:
                    inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                    hole_mask = hole_mask.to(DEVICE)
                    
                    outputs = model(inputs)
                    
                    out_real = outputs.clone()
                    tgt_real = targets[:, 0:2].clone()
                    
                    out_real[:, 0] = (out_real[:, 0] * b4_s) + b4_m
                    tgt_real[:, 0] = (tgt_real[:, 0] * b4_s) + b4_m
                    out_real[:, 1] = (out_real[:, 1] * b8_s) + b8_m
                    tgt_real[:, 1] = (tgt_real[:, 1] * b8_s) + b8_m
                    
                    raw_val = criterion(out_real, tgt_real)
                    
                    v_hole = (raw_val * hole_mask).sum() / (hole_mask.sum() + 1e-8)
                    val_hole_loss_real += v_hole.item()
            
            avg_train = train_loss / len(train_loader)
            avg_val_hole = val_hole_loss_real / len(val_loader)
            
            is_best = avg_val_hole < best_val_hole_loss
            if is_best:
                best_val_hole_loss = avg_val_hole
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'z_stats': dataset_stats
                }, "model_master_zscore_best.pth")
                print(f"✅ Epoch {epoch+1}: Train Z-Loss {avg_train:.4f} | Val Benchmark MAE (Real) {avg_val_hole:.4f} 🏆 NEW BEST!")
            else:
                print(f"✅ Epoch {epoch+1}: Train Z-Loss {avg_train:.4f} | Val Benchmark MAE (Real) {avg_val_hole:.4f}")
            
            # 🚨 Write empirical evidence to disk immediately
            history_writer.writerow([epoch+1, f"{avg_train:.6f}", f"{avg_val_hole:.6f}", is_best])
            history_file.flush()

    finally:
        # Guarantee the file closes cleanly even if training is manually interrupted
        history_file.close()
        print("✅ Ledger safely closed. Empirical evidence saved to training_history_master.csv")

if __name__ == "__main__":
    run_training()
    save_final_plot("training_history_master.csv")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import glob
import os
from tqdm import tqdm
from sklearn.metrics import r2_score, mean_squared_error

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
DATA_DIR = "3d_unet_grid_patches"
MODEL_PATH = "model_master_zscore_best.pth"
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_GRIDS = {3, 13}

SCENARIOS = [
    (0, 0), (1, 0), (0, 1), (1, 1), (2, 1), 
    (1, 2), (2, 2), (2, 3), (3, 2), (3, 3)
]

# ==========================================
# 📦 DATASET (Synchronized Mask Semantics)
# ==========================================
class ScenarioTestDataset(Dataset):
    def __init__(self, data_dir, stats, seq_len=16):
        self.seq_len = seq_len
        self.current_xl = 0
        self.current_xr = 0
        self.b4_m, self.b4_s, self.b8_m, self.b8_s = stats
        
        all_files = sorted(glob.glob(os.path.join(data_dir, "*.npz")))
        self.files = [f for f in all_files if int(os.path.basename(f).split('_')[1]) in TEST_GRIDS]
        
        print(f"📥 Caching {len(self.files)} test files...")
        self.cache = [np.load(f)['data'] for f in tqdm(self.files)]
        
        self.MIN_LAT, self.MAX_LAT = 49.0, 61.0

    def set_scenario(self, xl, xr):
        self.current_xl, self.current_xr = xl, xr

    def forward_fill_numpy(self, arr):
        mask = np.isnan(arr)
        out = arr.copy()
        for t in range(1, out.shape[0]):
            out[t][mask[t]] = out[t-1][mask[t]]
        mask_rev = np.isnan(out)
        for t in range(out.shape[0]-2, -1, -1):
            out[t][mask_rev[t]] = out[t+1][mask_rev[t]]
        return np.nan_to_num(out, nan=0.0)

    def __len__(self): return len(self.files) * 5

    def __getitem__(self, idx):
        actual_idx = idx % len(self.files)
        full_seq = self.cache[actual_idx]
        
        # 🚨 THE GUARD: Handle sequences shorter than seq_len (e.g., < 16 frames)
        if full_seq.shape[0] < self.seq_len:
            reps = int(np.ceil(self.seq_len / full_seq.shape[0]))
            full_seq = np.tile(full_seq, (reps, 1, 1, 1))
            
        total_time = full_seq.shape[0]
        start_t = np.random.randint(0, total_time - self.seq_len + 1)
        seq_raw = full_seq[start_t : start_t + self.seq_len].copy()
        
        # 1. Capture PHYSICAL TRUTH (Raw Validity)
        real_target_mask_np = ~np.isnan(seq_raw[:, :, :, 0:2])
        real_target_mask = torch.from_numpy(real_target_mask_np.copy()).permute(3, 0, 1, 2).float()

        # 2. Z-SCORE NORMALIZATION (Symmetric to Training)
        seq_raw[:, :, :, 0] = (seq_raw[:, :, :, 0] - self.b4_m) / (self.b4_s + 1e-8)
        seq_raw[:, :, :, 1] = (seq_raw[:, :, :, 1] - self.b8_m) / (self.b8_s + 1e-8)
        seq_raw[:, :, :, 18] = (seq_raw[:, :, :, 18] - self.MIN_LAT) / (self.MAX_LAT - self.MIN_LAT)
        
        # 3. INPUT PREPARATION (Imputed for Model Input Only)
        seq_filled = self.forward_fill_numpy(seq_raw)
        clean_input = torch.from_numpy(seq_filled).permute(3, 0, 1, 2).float()

        C, T, H, W = clean_input.shape
        cloud_mask = torch.ones((1, T, H, W))
        
        # 4. SCENARIO HOLE LOGIC (Fixed Difficulty Benchmark)
        center_t = T // 2
        hole_start = max(0, center_t - self.current_xl)
        hole_end = min(T, center_t + self.current_xr + 1)
        
        # Fixed 32x32 spatial blackout in the center
        size = 32
        top, left = (H - size) // 2, (W - size) // 2
        
        cloud_mask[:, hole_start:hole_end, top:top+size, left:left+size] = 0.0
        
        # 5. MODEL INPUT ASSEMBLY (20 Channels)
        model_input = clean_input.clone()
        # Blind the model to the reflectance inside the synthetic hole
        model_input[0:2][cloud_mask.repeat(2, 1, 1, 1) == 0.0] = 0.0
        
        # 🚨 TRUE Visibility Mask (Intersection of sensor truth and synthetic drop)
        natural_valid = (real_target_mask[0] * real_target_mask[1]).unsqueeze(0)
        input_mask_channel = natural_valid.clone()
        input_mask_channel[:, hole_start:hole_end, top:top+size, left:left+size] = 0.0
        
        final_input = torch.cat([model_input, input_mask_channel], dim=0)
        
        # 6. 🚨 TARGET IS PHYSICAL TRUTH (Raw Observed Data)
        target_np = np.ascontiguousarray(seq_raw[:, :, :, 0:2])
        target_tensor = torch.from_numpy(target_np).permute(3, 0, 1, 2).float()
        target_tensor = torch.nan_to_num(target_tensor, nan=0.0) 
        
        # 7. METRIC EVALUATION MASK (Center frame 'Y' + Real sensor data only)
        test_mask = torch.zeros((2, T, H, W))
        test_mask[:, center_t, top:top+size, left:left+size] = 1.0
        test_mask = test_mask * real_target_mask # Strict sensor validation
        
        return final_input, target_tensor, test_mask

# ==========================================
# 🧠 MODEL (20 Channels)
# ==========================================
class HybridUNet3D(nn.Module):
    def __init__(self, in_channels=20):
        super().__init__()
        self.input_norm = nn.BatchNorm3d(in_channels)
        self.enc1 = self.conv_block(in_channels, 32)
        self.pool1 = nn.MaxPool3d(2); self.enc2 = self.conv_block(32, 64)
        self.pool2 = nn.MaxPool3d(2); self.bottleneck = self.conv_block(64, 128)
        self.up2 = nn.ConvTranspose3d(128, 64, 2, 2); self.dec2 = self.conv_block(128, 64)
        self.up1 = nn.ConvTranspose3d(64, 32, 2, 2); self.dec1 = self.conv_block(64, 32)
        self.head_b4 = nn.Conv3d(32, 1, 1); self.head_b8 = nn.Conv3d(32, 1, 1)
    def conv_block(self, in_c, out_c):
        return nn.Sequential(nn.Conv3d(in_c, out_c, 3, 1, 1), nn.BatchNorm3d(out_c), nn.ReLU(),
                             nn.Conv3d(out_c, out_c, 3, 1, 1), nn.BatchNorm3d(out_c), nn.ReLU())
    def forward(self, x):
        e1 = self.enc1(self.input_norm(x)); p1 = self.pool1(e1)
        e2 = self.enc2(p1); p2 = self.pool2(e2); b = self.bottleneck(p2)
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.cat([self.head_b4(d1), self.head_b8(d1)], dim=1)

# ==========================================
# 📊 EXECUTION
# ==========================================
if __name__ == "__main__":
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
    model = HybridUNet3D(in_channels=20).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    stats = ckpt['z_stats']
    ds = ScenarioTestDataset(DATA_DIR, stats=stats)
    
    print(f"\n📈 3D U-NET SCENARIO TEST (Fully Synchronized)")
    print("-" * 105)
    
    for i, (xl, xr) in enumerate(SCENARIOS):
        ds.set_scenario(xl, xr)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
        t_vals, p_vals = {0:[], 1:[]}, {0:[], 1:[]}
        
        with torch.no_grad():
            for inputs, targets, test_mask in tqdm(loader, leave=False):
                inputs, targets, test_mask = inputs.to(DEVICE), targets.to(DEVICE), test_mask.to(DEVICE)
                outputs = model(inputs)
                for b in [0, 1]:
                    mask = test_mask[:, b] > 0.5
                    if mask.sum() > 0:
                        t_vals[b].append(targets[:,b][mask].cpu().numpy())
                        p_vals[b].append(outputs[:,b][mask].cpu().numpy())

        for b in [0, 1]:
            # Use the concatenated lists t_vals and p_vals populated in the batch loop
            if t_vals[b]:
                y_t = np.concatenate(t_vals[b])
                y_p = np.concatenate(p_vals[b])
                
                # 🚨 DENORMALIZE: stats = (b4_m, b4_s, b8_m, b8_s)
                m, s = (stats[0], stats[1]) if b == 0 else (stats[2], stats[3])
                y_t = (y_t * s) + m
                y_p = (y_p * s) + m
                
                # 🚨 PHYSICAL CONSTRAINT: Light reflectance cannot be negative
                y_p = np.maximum(y_p, 0.0) 
                
                # 📊 METRIC CALCULATION
                r2 = r2_score(y_t, y_p)
                rmse = np.sqrt(mean_squared_error(y_t, y_p))
                bias = np.mean(y_p - y_t)
                
                print(f"Scen {i+1} | Band {b+4} | R2: {r2_score(y_t, y_p):.4f} | RMSE: {np.sqrt(mean_squared_error(y_t, y_p)):.4f}")

In [ ]:
"""
Test Script for 3D U-Net Gap Filling
"""

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import glob
import os
import random
import csv
from tqdm import tqdm
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
DATA_DIR = "3d_unet_grid_patches"
MODEL_PATH = "model_master_zscore_best.pth"
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_GRIDS = {3, 13}
SEQ_LEN = 16
HOLE_SIZE = 32  # spatial extent of synthetic gap

# Gap scenarios: (frames_left, frames_right) relative to centre
# Total gap duration = xl + xr + 1
SCENARIOS = [
    (0, 0), (1, 0), (0, 1), (1, 1), (2, 1),
    (1, 2), (2, 2), (2, 3), (3, 2), (3, 3)
]

OUTPUT_CSV = "test_results_corrected.csv"


def seed_everything(seed=42):
    """Lock all random seeds for full reproducibility."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ==========================================
# 📦 DETERMINISTIC TEST DATASET
# ==========================================
class DeterministicTestDataset(Dataset):
    """
    Key differences from original:
    - Uses FIXED strided temporal windows (no randomness)
    - Each chip produces multiple samples via stride, exhaustively
    - Evaluates ALL frames inside the gap, not just the centre
    """

    def __init__(self, data_dir, stats, seq_len=16, stride=8):
        self.seq_len = seq_len
        self.stride = stride
        self.b4_m, self.b4_s, self.b8_m, self.b8_s = stats
        self.MIN_LAT, self.MAX_LAT = 49.0, 61.0

        # Current scenario (set before each evaluation)
        self.current_xl = 0
        self.current_xr = 0

        # Load test files
        all_files = sorted(glob.glob(os.path.join(data_dir, "*.npz")))
        self.files = [
            f for f in all_files
            if int(os.path.basename(f).split('_')[1]) in TEST_GRIDS
        ]
        print(f"📥 Caching {len(self.files)} test files...")
        self.cache = [np.load(f)['data'] for f in tqdm(self.files)]

        # Pre-compute all (file_idx, start_t) pairs deterministically
        self.samples = []
        for file_idx, chip in enumerate(self.cache):
            T = chip.shape[0]
            if T < seq_len:
                # Single padded sample for short sequences
                self.samples.append((file_idx, 0, True))
            else:
                # Strided windows covering the full temporal extent
                for start_t in range(0, T - seq_len + 1, stride):
                    self.samples.append((file_idx, start_t, False))
                # Ensure the last window is always included
                last_start = T - seq_len
                if (last_start % stride) != 0:
                    self.samples.append((file_idx, last_start, False))

        print(f"   Total deterministic samples: {len(self.samples)}")

    def set_scenario(self, xl, xr):
        self.current_xl = xl
        self.current_xr = xr

    def forward_fill_numpy(self, arr):
        mask = np.isnan(arr)
        out = arr.copy()
        for t in range(1, out.shape[0]):
            out[t][mask[t]] = out[t - 1][mask[t]]
        mask_rev = np.isnan(out)
        for t in range(out.shape[0] - 2, -1, -1):
            out[t][mask_rev[t]] = out[t + 1][mask_rev[t]]
        return np.nan_to_num(out, nan=0.0)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_idx, start_t, needs_pad = self.samples[idx]
        full_seq = self.cache[file_idx]

        # Handle short sequences
        if needs_pad:
            reps = int(np.ceil(self.seq_len / full_seq.shape[0]))
            full_seq = np.tile(full_seq, (reps, 1, 1, 1))
            start_t = 0

        seq_raw = full_seq[start_t: start_t + self.seq_len].copy()

        # 1. Capture physical truth (raw validity before any processing)
        real_target_mask_np = ~np.isnan(seq_raw[:, :, :, 0:2])
        real_target_mask = torch.from_numpy(
            real_target_mask_np.copy()
        ).permute(3, 0, 1, 2).float()

        # 2. Z-score normalisation (symmetric to training)
        seq_raw[:, :, :, 0] = (seq_raw[:, :, :, 0] - self.b4_m) / (self.b4_s + 1e-8)
        seq_raw[:, :, :, 1] = (seq_raw[:, :, :, 1] - self.b8_m) / (self.b8_s + 1e-8)
        seq_raw[:, :, :, 18] = (
            (seq_raw[:, :, :, 18] - self.MIN_LAT) / (self.MAX_LAT - self.MIN_LAT)
        )

        # 3. Input preparation (imputed for model input only)
        seq_filled = self.forward_fill_numpy(seq_raw)
        clean_input = torch.from_numpy(seq_filled).permute(3, 0, 1, 2).float()

        C, T, H, W = clean_input.shape
        cloud_mask = torch.ones((1, T, H, W))

        # 4. Scenario hole logic (fixed spatial centre)
        center_t = T // 2
        hole_start = max(0, center_t - self.current_xl)
        hole_end = min(T, center_t + self.current_xr + 1)

        top = (H - HOLE_SIZE) // 2
        left = (W - HOLE_SIZE) // 2

        cloud_mask[:, hole_start:hole_end, top:top + HOLE_SIZE, left:left + HOLE_SIZE] = 0.0

        # 5. Model input assembly (20 channels)
        model_input = clean_input.clone()
        model_input[0:2][cloud_mask.repeat(2, 1, 1, 1) == 0.0] = 0.0

        natural_valid = (real_target_mask[0] * real_target_mask[1]).unsqueeze(0)
        input_mask_channel = natural_valid.clone()
        input_mask_channel[:, hole_start:hole_end, top:top + HOLE_SIZE, left:left + HOLE_SIZE] = 0.0

        final_input = torch.cat([model_input, input_mask_channel], dim=0)

        # 6. Target (z-scored, NaN filled with 0 for tensor compat)
        target_np = np.ascontiguousarray(seq_raw[:, :, :, 0:2])
        target_tensor = torch.from_numpy(target_np).permute(3, 0, 1, 2).float()
        target_tensor = torch.nan_to_num(target_tensor, nan=0.0)

        # 7. FIX: Evaluate ALL frames in the gap, not just the centre
        full_gap_mask = torch.zeros((2, T, H, W))
        full_gap_mask[:, hole_start:hole_end, top:top + HOLE_SIZE, left:left + HOLE_SIZE] = 1.0
        full_gap_mask = full_gap_mask * real_target_mask  # only where sensor had real data

        # Also provide centre-only mask for backward compatibility
        centre_only_mask = torch.zeros((2, T, H, W))
        centre_only_mask[:, center_t, top:top + HOLE_SIZE, left:left + HOLE_SIZE] = 1.0
        centre_only_mask = centre_only_mask * real_target_mask

        return final_input, target_tensor, full_gap_mask, centre_only_mask


# ==========================================
# 🧠 MODEL (must match training architecture exactly)
# ==========================================
class HybridUNet3D(nn.Module):
    def __init__(self, in_channels=20):
        super().__init__()
        self.input_norm = nn.BatchNorm3d(in_channels)
        self.enc1 = self.conv_block(in_channels, 32)
        self.pool1 = nn.MaxPool3d(2)
        self.enc2 = self.conv_block(32, 64)
        self.pool2 = nn.MaxPool3d(2)
        self.bottleneck = self.conv_block(64, 128)
        self.up2 = nn.ConvTranspose3d(128, 64, 2, 2)
        self.dec2 = self.conv_block(128, 64)
        self.up1 = nn.ConvTranspose3d(64, 32, 2, 2)
        self.dec1 = self.conv_block(64, 32)
        self.head_b4 = nn.Conv3d(32, 1, 1)
        self.head_b8 = nn.Conv3d(32, 1, 1)

    def conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, 1, 1), nn.BatchNorm3d(out_c), nn.ReLU(),
            nn.Conv3d(out_c, out_c, 3, 1, 1), nn.BatchNorm3d(out_c), nn.ReLU()
        )

    def forward(self, x):
        e1 = self.enc1(self.input_norm(x))
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        b = self.bottleneck(p2)
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.cat([self.head_b4(d1), self.head_b8(d1)], dim=1)


# ==========================================
# 📊 METRIC COMPUTATION
# ==========================================
def compute_all_metrics(y_true, y_pred):
    """
    Compute all 5 metrics on physical (denormalized) reflectance values.
    Returns dict with: RMSE, MAE, Bias, Pearson_R, R2
    """
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    bias = float(np.mean(y_pred - y_true))
    r2 = float(r2_score(y_true, y_pred))

    # Pearson R (handle edge case of zero variance)
    if np.std(y_true) < 1e-10 or np.std(y_pred) < 1e-10:
        pearson_r = 0.0
    else:
        pearson_r = float(pearsonr(y_true, y_pred)[0])

    return {
        'RMSE': rmse,
        'MAE': mae,
        'Bias': bias,
        'Pearson_R': pearson_r,
        'R2': r2
    }


def denormalize(values, band_idx, stats):
    """Convert z-scored values back to physical reflectance."""
    b4_m, b4_s, b8_m, b8_s = stats
    if band_idx == 0:
        return (values * b4_s) + b4_m
    else:
        return (values * b8_s) + b8_m


# ==========================================
# 🚀 MAIN EXECUTION
# ==========================================
def run_test():
    seed_everything(42)

    # Load model
    ckpt = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    model = HybridUNet3D(in_channels=20).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    stats = ckpt['z_stats']
    ds = DeterministicTestDataset(DATA_DIR, stats=stats, seq_len=SEQ_LEN, stride=8)

    # Prepare CSV output
    csv_file = open(OUTPUT_CSV, 'w', newline='')
    writer = csv.writer(csv_file)
    writer.writerow([
        'Scenario', 'XL', 'XR', 'Gap_Frames', 'Eval_Region',
        'Band', 'N_Pixels',
        'RMSE_raw', 'MAE_raw', 'Bias_raw', 'Pearson_R_raw', 'R2_raw',
        'RMSE_clipped', 'MAE_clipped', 'Bias_clipped', 'Pearson_R_clipped', 'R2_clipped',
        'Pct_Negative_Preds'
    ])

    band_names = {0: 'B4_Red', 1: 'B8_NIR'}

    print(f"\n{'='*120}")
    print(f"  DETERMINISTIC GAP-FILLING EVALUATION — All 5 Metrics, Full Gap + Centre-Only")
    print(f"{'='*120}")

    for scen_idx, (xl, xr) in enumerate(SCENARIOS):
        gap_frames = xl + xr + 1
        ds.set_scenario(xl, xr)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        # Accumulators: [band][eval_region] -> lists
        collectors = {
            b: {'full_gap': {'t': [], 'p': []}, 'centre': {'t': [], 'p': []}}
            for b in [0, 1]
        }

        with torch.no_grad():
            for inputs, targets, full_gap_mask, centre_mask in tqdm(
                loader, desc=f"Scenario ({xl},{xr})", leave=False
            ):
                inputs = inputs.to(DEVICE)
                targets = targets.to(DEVICE)
                full_gap_mask = full_gap_mask.to(DEVICE)
                centre_mask = centre_mask.to(DEVICE)

                outputs = model(inputs)

                for b in [0, 1]:
                    # Full gap evaluation
                    fg_mask = full_gap_mask[:, b] > 0.5
                    if fg_mask.sum() > 0:
                        collectors[b]['full_gap']['t'].append(
                            targets[:, b][fg_mask].cpu().numpy()
                        )
                        collectors[b]['full_gap']['p'].append(
                            outputs[:, b][fg_mask].cpu().numpy()
                        )

                    # Centre-only evaluation
                    cm = centre_mask[:, b] > 0.5
                    if cm.sum() > 0:
                        collectors[b]['centre']['t'].append(
                            targets[:, b][cm].cpu().numpy()
                        )
                        collectors[b]['centre']['p'].append(
                            outputs[:, b][cm].cpu().numpy()
                        )

        # Compute and report metrics
        print(f"\n  Scenario {scen_idx+1}: gap=({xl}L, {xr}R) = {gap_frames} frames")
        print(f"  {'-'*110}")

        for b in [0, 1]:
            for region_name in ['full_gap', 'centre']:
                t_list = collectors[b][region_name]['t']
                p_list = collectors[b][region_name]['p']

                if not t_list:
                    print(f"    {band_names[b]} | {region_name:>9s} | NO VALID PIXELS")
                    continue

                y_t_z = np.concatenate(t_list)
                y_p_z = np.concatenate(p_list)

                # Denormalize to physical reflectance
                y_t = denormalize(y_t_z, b, stats)
                y_p = denormalize(y_p_z, b, stats)

                n_pixels = len(y_t)
                pct_negative = float((y_p < 0).sum()) / n_pixels * 100

                # RAW metrics (no clipping)
                metrics_raw = compute_all_metrics(y_t, y_p)

                # CLIPPED metrics (reflectance >= 0)
                y_p_clip = np.maximum(y_p, 0.0)
                metrics_clip = compute_all_metrics(y_t, y_p_clip)

                # Write to CSV
                writer.writerow([
                    scen_idx + 1, xl, xr, gap_frames, region_name,
                    band_names[b], n_pixels,
                    f"{metrics_raw['RMSE']:.6f}", f"{metrics_raw['MAE']:.6f}",
                    f"{metrics_raw['Bias']:.6f}", f"{metrics_raw['Pearson_R']:.6f}",
                    f"{metrics_raw['R2']:.6f}",
                    f"{metrics_clip['RMSE']:.6f}", f"{metrics_clip['MAE']:.6f}",
                    f"{metrics_clip['Bias']:.6f}", f"{metrics_clip['Pearson_R']:.6f}",
                    f"{metrics_clip['R2']:.6f}",
                    f"{pct_negative:.2f}"
                ])
                csv_file.flush()

                # Console output
                label = "FULL GAP" if region_name == 'full_gap' else " CENTRE "
                print(
                    f"    {band_names[b]} | {label} | n={n_pixels:>8d} | "
                    f"RMSE={metrics_raw['RMSE']:.4f}  MAE={metrics_raw['MAE']:.4f}  "
                    f"Bias={metrics_raw['Bias']:+.4f}  R={metrics_raw['Pearson_R']:.4f}  "
                    f"R²={metrics_raw['R2']:.4f}  |  neg={pct_negative:.1f}%"
                )

    csv_file.close()
    print(f"\n{'='*120}")
    print(f"  ✅ Results saved to {OUTPUT_CSV}")
    print(f"{'='*120}")


if __name__ == "__main__":
    run_test()